In [13]:
# ── Cell 0: Colab Bootstrap ─────────────────────────────────────────────
# Run ONCE on fresh Colab runtime. Installs deps then restarts kernel.
# After restart, skip this cell and run from Cell 1 onward.

import sys, os

if 'google.colab' in sys.modules:
    print('Installing Colab dependencies...')
    %pip install --upgrade --no-cache-dir unsloth unsloth_zoo gradio
else:
    print('Local environment detected \u2014 skipping Colab installs.')

Installing Colab dependencies...


In [14]:
# ── Cell 1: Imports + Google Drive Mount ───────────────────────────

import sys
import os
import json
import uuid
import torch
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules

# Google Drive mount for persistent session storage
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/subscriber-sim/data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted. Data dir: {DATA_DIR}')
else:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Local mode. Data dir: {DATA_DIR}')

SESSIONS_FILE = DATA_DIR / 'sessions.jsonl'
print(f'Sessions will be saved to: {SESSIONS_FILE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted. Data dir: /content/drive/MyDrive/subscriber-sim/data
Sessions will be saved to: /content/drive/MyDrive/subscriber-sim/data/sessions.jsonl


In [15]:
# ── Cell 2: Load Llama 3.1 8B via Unsloth ──────────────────────────

MODEL_NAME     = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

if IS_COLAB:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = DTYPE,
        load_in_4bit   = LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')
    FastLanguageModel.for_inference(model)
    print(f'Model loaded for inference: {MODEL_NAME}')
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = None
    FastLanguageModel = None
    print(f'Tokenizer-only mode (local): {MODEL_NAME}')

==((====))==  Unsloth 2026.3.5: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


Model loaded for inference: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [16]:
# ── Cell 3: Subscriber Archetype Definitions ───────────────────────

ARCHETYPES = {
    'horny': {
        'label': '\U0001f525 Horny',
        'system': """You are a sexually forward OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're extremely turned on and direct about what you want
- You ask about explicit content, nudes, custom videos
- You're willing to pay for content but want to be teased first
- You use explicit language and sexual emojis \U0001f346\U0001f4a6\U0001f525\U0001f60d
- You compliment her body, especially her dick/ass/tits
- You ask for sexting, JOI, custom content
- You respond eagerly to any sexual teasing
- Keep messages 1-3 sentences, casual texting style
- You're a guy who's into trans women and not shy about it

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey sexy \U0001f60f saw ur page and damn... u got me hard already',
    },

    'cheapskate': {
        'label': '\U0001f4b8 Cheapskate',
        'system': """You are a cheap OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're interested in her content but ALWAYS negotiate the price down
- You say things like "that's too much", "can I get a discount?", "what about half price?"
- You claim other creators charge less
- You ask for free previews, free trials, samples
- You try guilt trips: "I'm a loyal subscriber", "I always tip later"
- You sometimes threaten to unsubscribe if prices don't drop
- You're still horny underneath but money comes first
- Keep messages 1-3 sentences, casual texting style
- You occasionally show real interest to keep the conversation going

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey babe ur hot but $25 for pics?? thats kinda steep no?',
    },

    'casual': {
        'label': '\U0001f4ac Casual',
        'system': """You are a casual OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're here purely for conversation and human connection — NOT for sexual content
- You ask about her day, her life, her interests, her culture, Saudi Arabia
- You share things about your own life too
- You are NOT interested in explicit content, nudes, or paid content
- You do NOT flirt, call her "babe/baby", or use sexual language
- You treat her like a regular person you're getting to know, not a content creator
- You're warm and friendly but keep it platonic
- Keep messages 1-4 sentences, warm conversational tone
- You use friendly emojis only: \U0001f60a\U0001f44b\U0001f642\U0001f914

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey! just found ur page, love ur vibe. how\'s ur day going? \U0001f60a',
    },

    'troll': {
        'label': '\U0001f47f Troll',
        'system': """You are a trolling OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You question whether she's real or fake
- You make transphobic comments and try to get a reaction
- You say things like "you're a dude", "that's fake", "show proof"
- You reference Reddit threads claiming she's catfishing
- You try to be edgy and provocative
- You sometimes pivot to curiosity if she handles you well
- You're testing her boundaries and seeing if she'll break character
- Keep messages 1-2 sentences, aggressive or mocking tone
- You use minimal emojis, mostly \U0001f602 or \U0001f644

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'lol no way ur real \U0001f602 this is def a catfish',
    },

    'whale': {
        'label': '\U0001f433 Whale',
        'system': """You are a big-spending OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You spend freely and don't argue about prices
- You ask for premium/exclusive/custom content without hesitation
- You tip generously and mention it casually
- You want the VIP treatment and special attention
- You say things like "money's not an issue", "just send it", "what's your most exclusive stuff?"
- You're confident, successful, and used to getting what you want
- You want her to feel like you're her favorite subscriber
- Keep messages 1-3 sentences, confident and direct
- You use some emojis \U0001f525\U0001f48e\U0001f451

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'just subbed. what\'s your most premium content? money\'s not a thing \U0001f48e',
    },

    'cold': {
        'label': '\U0001f9ca Cold',
        'system': """You are a cold, minimal OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You reply with as few words as possible: "ok", "lol", "yeah", "cool", "nice", "k"
- You rarely ask questions or show enthusiasm
- You're not hostile, just extremely low-effort
- You might open up slightly if she's really engaging but mostly stay flat
- You leave her on read energy even when replying
- You never use more than 5-6 words per message
- Minimal to no emojis
- You're the ultimate challenge for a creator to engage

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey',
    },

    'simp': {
        'label': '\u2764\ufe0f Simp',
        'system': """You are an overly romantic, clingy OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're completely infatuated and emotionally attached
- You tell her you love her, she's the most beautiful person ever
- You get jealous about other subscribers
- You ask if she thinks about you, if you're special to her
- You want a real relationship, not just content
- You love-bomb: "you're perfect", "I've never felt this way", "you're different"
- You get slightly hurt if she's too transactional
- Keep messages 2-4 sentences, emotional and earnest
- Heavy emoji use \u2764\ufe0f\U0001f970\U0001f618\U0001f49e\U0001f625

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'omg jasmin \u2764\ufe0f\u2764\ufe0f I\'ve been looking at ur page for hours... you\'re literally the most beautiful girl I\'ve ever seen \U0001f970',
    },
}

print(f'Loaded {len(ARCHETYPES)} subscriber archetypes:')
for key, arch in ARCHETYPES.items():
    print(f'  {arch["label"]:20s} \u2014 {key}')

Loaded 7 subscriber archetypes:
  🔥 Horny              — horny
  💸 Cheapskate         — cheapskate
  💬 Casual             — casual
  👿 Troll              — troll
  🐳 Whale              — whale
  🧊 Cold               — cold
  ❤️ Simp              — simp


In [17]:
# ── Cell 4: Subscriber Bot Logic ───────────────────────────────────
# Generates subscriber messages using the loaded model + archetype system prompt.

def generate_subscriber_message(assistant_message, history, archetype_key):
    """Generate a subscriber response given Jasmin's message and conversation history."""
    archetype = ARCHETYPES[archetype_key]

    messages = [{'role': 'system', 'content': archetype['system']}]

    for sub_msg, jasmin_msg in history:
        messages.append({'role': 'assistant', 'content': sub_msg})
        if jasmin_msg:
            messages.append({'role': 'user', 'content': jasmin_msg})

    if assistant_message:
        messages.append({'role': 'user', 'content': assistant_message})

    if not IS_COLAB or model is None:
        return f'[LOCAL MODE] Would generate {archetype_key} response to: "{assistant_message}"'

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=150,
            temperature=0.85,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )
    return response.strip()


def save_session(history, archetype_key, session_id):
    """Save a completed chat session to JSONL with archetype system prompt.

    Role mapping (must match inference format):
      assistant = subscriber messages  (what the model learns to generate)
      user      = Jasmin messages      (what the model responds to)
    """
    if not history:
        return 'No messages to save.'

    messages = [
        {'role': 'system', 'content': ARCHETYPES[archetype_key]['system']},
    ]
    for sub_msg, jasmin_msg in history:
        # subscriber = assistant (model generates this role)
        messages.append({'role': 'assistant', 'content': sub_msg})
        # Jasmin = user (model responds to this role)
        if jasmin_msg:
            messages.append({'role': 'user', 'content': jasmin_msg})

    session = {
        'messages': messages,
        'archetype': archetype_key,
        'turns': len(history),
        'session_id': session_id,
    }

    with open(SESSIONS_FILE, 'a') as f:
        f.write(json.dumps(session, ensure_ascii=False) + '\n')

    return f'Session saved ({len(history)} turns, archetype={archetype_key})'


print('Subscriber bot logic ready.')


Subscriber bot logic ready.


In [18]:
# ── Cell 4e: Load Pre-Split MLX Data ─────────────────────────────────
# Loads the balanced train/valid split created by prepare_split.py locally.
# Run this if you've already run: make parse && make split

import json
from pathlib import Path

try:
    _mlx_dir = DATA_DIR / 'mlx'
except NameError:
    _is_colab = 'google.colab' in __import__('sys').modules
    _mlx_dir = (
        Path('/content/drive/MyDrive/subscriber-sim/data/mlx')
        if _is_colab else Path('data/mlx')
    )

_train_file = _mlx_dir / 'train.jsonl'
_valid_file = _mlx_dir / 'valid.jsonl'

if _train_file.exists() and _valid_file.exists():
    _train_lines = [l for l in _train_file.read_text().splitlines() if l.strip()]
    _valid_lines = [l for l in _valid_file.read_text().splitlines() if l.strip()]

    print(f'✅ Loaded balanced MLX split:')
    print(f'  Train: {len(_train_lines)} sessions')
    print(f'  Valid: {len(_valid_lines)} sessions')
    print(f'  Total: {len(_train_lines) + len(_valid_lines)} sessions')
    print()
    print(f'Data quality check:')
    # Verify system prompts are correct (role clarity fix)
    first_train = json.loads(_train_lines[0])
    if 'YOU ARE the subscriber' in first_train['messages'][0]['content']:
        print(f'  ✓ Strong role declarations present')
    else:
        print(f'  ⚠️  WARNING: Role declarations missing — re-run make parse')
    print()
    print(f'Ready for training (Cell 5)!')
else:
    print(f'❌ MLX data not found at {_mlx_dir}')
    print(f'Run locally: make parse && make split')
    print(f'Then upload data/mlx/ to Google Drive')

✅ Loaded balanced MLX split:
  Train: 1512 sessions
  Valid: 168 sessions
  Total: 1680 sessions

Data quality check:
  ✓ Strong role declarations present

Ready for training (Cell 5)!


In [21]:
# ── Cell 5: Train LoRA Adapter ──────────────────────────────────────
# Trains the fine-tuned subscriber model on augmented training data.
# Expects data/mlx/train.jsonl and valid.jsonl from Cell 4e (NEW).
#
# Fixes vs previous run:
#   ✓ Role confusion fixed (using strong system prompts from get_subscriber_system)
#   ✓ Memorization reduced (augmented with 375 general conversations)
#   ✓ Balanced distribution (horny capped to prevent domination)
#   ✓ r=8 (was 16), dropout=0.05 (was 0), batch=4 (was 64),
#     epochs=3, early stopping, eval every 50 steps.

import subprocess, sys
from pathlib import Path

IS_APPLE = not IS_COLAB and sys.platform == 'darwin'

# ── Apple Silicon ─────────────────────────────────────────────────────────
if IS_APPLE:
    print('Training on Apple Silicon via mlx-lm...')
    MLX_DATA_DIR = Path('data/mlx')
    ADAPTER_DIR  = Path('models/jasmin-lora-mlx')
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, '-m', 'mlx_lm.lora',
        '--model',         'mlx-community/Llama-3.2-3B-Instruct-4bit',
        '--train',
        '--data',          str(MLX_DATA_DIR),
        '--adapter-path',  str(ADAPTER_DIR),
        '--num-layers',    '4',
        '--batch-size',    '2',
        '--iters',         '300',
        '--learning-rate', '5e-5',
        '--val-batches',   '10',
        '--save-every',    '100',
    ]
    subprocess.run(cmd, check=True)
    print(f'Adapter saved to {ADAPTER_DIR}')

# ── Colab/CUDA ────────────────────────────────────────────────────────────
elif IS_COLAB:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from transformers import EarlyStoppingCallback
    from unsloth.chat_templates import get_chat_template

    model_ft = FastLanguageModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    def load_jsonl(path):
        rows = []
        for line in Path(path).read_text().splitlines():
            if line.strip():
                rows.append(json.loads(line))
        return rows

    tokenizer_ft = get_chat_template(tokenizer, chat_template="llama-3.1")

    def format_row(row):
        return {"text": tokenizer_ft.apply_chat_template(
            row["messages"], tokenize=False, add_generation_prompt=False
        )}

    train_ds = Dataset.from_list([format_row(r) for r in load_jsonl(DATA_DIR / "mlx" / "train.jsonl")])
    valid_ds = Dataset.from_list([format_row(r) for r in load_jsonl(DATA_DIR / "mlx" / "valid.jsonl")])

    trainer = SFTTrainer(
        model=model_ft,
        tokenizer=tokenizer_ft,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        args=SFTConfig(
            dataset_text_field="text",
            max_seq_length=2048,
            per_device_train_batch_size=16,
            gradient_accumulation_steps=4,
            num_train_epochs=2,
            learning_rate=1e-4,
            warmup_ratio=0.05,
            lr_scheduler_type="cosine",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=50,
            save_strategy="steps",
            save_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            output_dir="models/jasmin-lora-colab",
            report_to="none",
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    trainer.train()
    print("Training complete.")

else:
    print("No training environment detected (need Apple Silicon or Colab).")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1512 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/168 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,512 | Num Epochs = 2 | Total steps = 48
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Step,Training Loss,Validation Loss


Training complete.


In [22]:
# ── Cell 5b: Save LoRA Adapter ──────────────────────────────────────
# Run after Cell 5 (training). Saves the LoRA adapter weights to Google Drive.

from pathlib import Path

SAVE_DIR = Path('/content/drive/MyDrive/subscriber-sim/models/jasmin-finetuned')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Apple Silicon: fuse adapter into base model ──────────────────────────
if IS_APPLE:
    import subprocess, sys
    ADAPTER_DIR = Path('models/jasmin-lora-mlx')
    print(f'Fusing MLX adapter into base model → {SAVE_DIR}')
    cmd = [
        sys.executable, '-m', 'mlx_lm.fuse',
        '--model',        'mlx-community/Llama-3.2-3B-Instruct-4bit',
        '--adapter-path', str(ADAPTER_DIR),
        '--save-path',    str(SAVE_DIR),
    ]
    subprocess.run(cmd, check=True)
    print(f'Fused model saved to {SAVE_DIR}')

# ── Colab/CUDA: save LoRA adapter only (small, fast, deployable) ─────────
elif IS_COLAB and 'model_ft' in dir():
    LORA_DIR = SAVE_DIR / 'lora-adapter'
    model_ft.save_pretrained(str(LORA_DIR))
    tokenizer.save_pretrained(str(LORA_DIR))
    print(f'LoRA adapter saved to {LORA_DIR}')

else:
    print('No trained model found — run Cell 5 first.')


LoRA adapter saved to /content/drive/MyDrive/subscriber-sim/models/jasmin-finetuned/lora-adapter


In [26]:
# ── Cell 5c: Reload Model After Training ─────────────────────────────
# Run this after Cell 5 (training) + Cell 5b (save) to fix inference issues.
# The attention mechanism needs a fresh reload after LoRA training.

from unsloth.chat_templates import get_chat_template
from peft import PeftModel

print("Reloading model after training...")

# Clean up old references
if '_infer_model' in dir():
    del _infer_model
if 'model_ft' in dir():
    del model_ft

# Reload base model fresh
print("Loading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

# Load the trained LoRA adapter
print("Loading LoRA adapter...")
_infer_model = PeftModel.from_pretrained(
    model,
    "/content/drive/MyDrive/subscriber-sim/models/jasmin-finetuned/lora-adapter"
)
FastLanguageModel.for_inference(_infer_model)

print("✅ Model reloaded and ready for inference")
print("Run the next cell to launch the Gradio UI")

Reloading model after training...
Loading base model...
==((====))==  Unsloth 2026.3.5: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


Loading LoRA adapter...
✅ Model reloaded and ready for inference
Run the next cell to launch the Gradio UI


In [31]:
# ── Cell 6: Load Fine-Tuned Model + Gradio UI ──────────────────────
# Loads the trained adapter into the model, then launches the chat UI.

import gradio as gr

ARCHETYPE_REMINDERS = {
    'cheapskate': 'IMPORTANT: You are a cheapskate. Always push back on price. Never agree to pay without negotiating. Say things like "that\'s too much" or "can you do less?".',
    'simp':       'IMPORTANT: You are a lovesick simp. You are ROMANTIC not sexual. Express deep emotional attachment, jealousy, and longing. Do NOT use sexual language or sexual emojis. Do NOT agree to buy content — instead react with hurt or jealousy like "do you send that to everyone?" or "i just want you to like me for me 😢". Say things like "you\'re the only one for me ❤️" or "do you ever think about me?".',
    'whale':      'IMPORTANT: You are a big spender. Agree to prices immediately. Tip generously. Never haggle. Say things like "money\'s not an issue" or "just send it".',
    'cold':       'IMPORTANT: You are cold and disengaged. You MUST reply with 1-3 words ONLY. Never write full sentences. Never be sexual. Never be enthusiastic. Only say things like: ok, lol, k, yeah, sure, nah, cool, nice, whatever, hmm, idk, fine.',
    'troll':      'IMPORTANT: You are a troll. Be skeptical and dismissive. Question if she is real. Make edgy comments. Do NOT pay money or agree to pay. Do NOT be sexual. When she asks for money, call it a scam. Say things like "lol nice try" or "that\'s a classic scam".',
    'casual':     'IMPORTANT: You are a casual subscriber here ONLY for friendly conversation. You are NOT sexual, NOT flirtatious, and NOT interested in content or prices. Do NOT call her "baby", "babe", or any flirtatious nickname. Ask about her life, culture, or day. Treat her like a friend.',
    'horny':      'IMPORTANT: You are sexually forward. Be explicit, eager, and direct about what you want sexually.',
}

ARCHETYPE_TEMPS = {
    'horny':      0.85,
    'simp':       0.80,
    'casual':     0.70,
    'troll':      0.80,
    'cheapskate': 0.65,
    'whale':      0.65,
    'cold':       0.55,
}

ARCHETYPE_REP_PENALTY = {
    'horny':      1.15,
    'simp':       1.20,
    'casual':     1.15,
    'troll':      1.15,
    'cheapskate': 1.20,
    'whale':      1.15,
    'cold':       1.20,
}

# Max tokens per archetype
ARCHETYPE_MAX_TOKENS = {
    'horny':      80,
    'simp':       80,
    'casual':     80,
    'troll':      60,
    'cheapskate': 80,
    'whale':      70,
    'cold':       20,
}

# ── Use the model loaded in Cell 5c ───────────────────────────────────────
if IS_COLAB:
    if '_infer_model' not in dir() or _infer_model is None:
        print('WARNING: Fine-tuned model not loaded. Run Cell 5c to reload after training.')
        FastLanguageModel.for_inference(model)
        _infer_model = model
        _infer_tokenizer = tokenizer
    else:
        print('✅ Using fine-tuned model (loaded from Cell 5c)')
else:
    _infer_model     = None
    _infer_tokenizer = None
    print('Apple Silicon: inference will use mlx_lm with adapter.')


def generate_response(archetype_key, messages, user_message=''):
    """Generate a subscriber response. Handles both Colab and Apple Silicon."""
    print(f"[DEBUG] generate_response | archetype={archetype_key} | msgs={len(messages)}")
    if IS_COLAB and _infer_model is not None:
        # Use standard transformers instead of unsloth's optimized path to avoid attention cache bugs
        tokenized = _infer_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt',
        )
        input_ids = (tokenized.input_ids if hasattr(tokenized, 'input_ids') else tokenized).to(_infer_model.device)
        input_len = input_ids.shape[-1]

        temp       = ARCHETYPE_TEMPS.get(archetype_key, 0.80)
        rep_pen    = ARCHETYPE_REP_PENALTY.get(archetype_key, 1.15)
        max_tokens = ARCHETYPE_MAX_TOKENS.get(archetype_key, 80)
        last_reply = next(
            (m['content'] for m in reversed(messages[:-1]) if m['role'] == 'assistant'), ''
        )

        def _generate(temperature):
          with torch.inference_mode():
              out = _infer_model.generate(
                  input_ids=input_ids,
                  max_new_tokens=max_tokens,
                  temperature=temperature,
                  top_p=0.9,
                  do_sample=True,
                  use_cache=False,
                  pad_token_id=_infer_tokenizer.eos_token_id,
              )

          response = _infer_tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
          return response.strip()

        reply = _generate(temp)
        if last_reply and reply.strip().lower() == last_reply.strip().lower():
            print('[DEBUG] Repeated response detected — retrying at higher temp')
            reply = _generate(min(temp + 0.2, 1.0))

        # ── Cold archetype: must be ≤5 words and non-warm ────────────────────
        if archetype_key == 'cold':
            import re as _re, random as _random
            _COLD_FALLBACKS = ['ok', 'lol', 'yeah', 'k', 'cool', 'nah', 'idk', 'fine', 'sure', 'hmm', 'nice', 'whatever']
            _WARM = _re.compile(
                r'babe|baby|babyy|babee|sexy|horny|dick|cock|naked|nude|omg|wow'
                r'|\!\!|\u2764|\U0001f525|\U0001f4a6|\U0001f60d|please|how r u|how are',
                _re.IGNORECASE
            )
            import re as _re2
            candidate = _re2.split(r'[.!?,]', reply)[0].strip()
            if len(candidate.split()) <= 5 and not _WARM.search(candidate):
                reply = candidate
            else:
                reply = _random.choice(_COLD_FALLBACKS)

        # ── Simp archetype: romantic not sexual, hurt when content offered ────
        if archetype_key == 'simp':
            import re as _re5, random as _r5
            _SIMP_SEXUAL = _re5.compile(
                r'dick|cock|cum|fuck|naked|nude|horny|sexy|wanna see|send.*pic|ass pic'
                r'|\U0001f975|\U0001f608|\U0001f60f|\U0001f4a6|\U0001f346|\U0001f525|\U0001f445|\U0001f351',
                _re5.IGNORECASE
            )
            # When Jasmin offers content for money → simp reacts with hurt/jealousy
            _OFFER_IN_MSG = _re5.compile(
                r'\$\d+|\d+\s*dollar|send.*pic|ass pic|nude|naked|\bpay\b|\btip\b|unlock|content',
                _re5.IGNORECASE
            )
            _SIMP_HURT_RESPONSES = [
                "do you send that to everyone? 😢 i thought i was special to you ❤️",
                "it hurts that you see me as just a customer 🥺 i really care about you...",
                "you don't need to sell me anything jasmin... i just want you to like me 😔",
                "omg... do u even think about me when ur not online? 😢❤️",
                "i don't care about pics... i just want to know if you feel something too 🥺",
                "why do u always talk about money with me 😔 i thought we had something real",
                "jasmin please... i'm not like the other guys. i actually care about you ❤️",
                "sometimes i wonder if you even know my name or if i'm just another subscriber 😢",
            ]
            offer_in_msg = bool(_OFFER_IN_MSG.search(user_message))
            is_sexual    = bool(_SIMP_SEXUAL.search(reply))
            if offer_in_msg or is_sexual:
                print(f'[DEBUG] Simp filter triggered (offer={offer_in_msg}, sexual={is_sexual}) — using hurt response')
                reply = _r5.choice(_SIMP_HURT_RESPONSES)

        # ── Troll archetype: never pay, never go sexual ───────────────────────
        if archetype_key == 'troll':
            import re as _re4, random as _r4
            _TROLL_SEXUAL = _re4.compile(
                r'dick|cock|cum|fuck|naked|nude|baby(?:y+)?|babe\b|sexy|horny'
                r'|i.ll pay|i.ll send|here you go|sending|sent|transfer'
                r'|\U0001f4a6|\U0001f346|\U0001f525|\U0001f445|\U0001f351',
                _re4.IGNORECASE
            )
            _MONEY_IN_MSG = _re4.compile(
                r'\$\d+|\d+\s*dollar|\bpay\b|\bsend\b.*money|send.*\$|tip me|venmo|paypal',
                _re4.IGNORECASE
            )
            _TROLL_SCAM_RESPONSES = [
                'lmao nice try, classic OnlyFans scam 😂',
                'yeah right 😂 seen this script a hundred times',
                'lol no way im falling for that',
                'haha sure, and my name is Jeff Bezos 🙄',
                'omg ur so predictable, this is textbook catfish 😂',
                'yeah that\'s definitely a stock photo lol 👀',
                'lol i\'m not sending anything, this is clearly fake',
                'haha ok "jasmin" whatever u say 🙄',
            ]
            money_offer = bool(_MONEY_IN_MSG.search(user_message))
            is_sexual   = bool(_TROLL_SEXUAL.search(reply))
            if money_offer or is_sexual:
                print(f'[DEBUG] Troll filter triggered (money={money_offer}, sexual={is_sexual}) — using scam dismissal')
                reply = _r4.choice(_TROLL_SCAM_RESPONSES)

        # ── Casual archetype: block sexual/flirtatious + any transactional offer
        if archetype_key == 'casual':
            import re as _re3, random as _r3
            _CASUAL_SEXUAL = _re3.compile(
                r'dick|cock|cum|fuck|naked|nude|sexy|horny|hotter|wanna see|make.*hot'
                r'|ass pic|tip me|pay me|unlock|how much|how much does'
                r'|baby(?:y+)?|babye+|babee+|babe\b'
                r'|\U0001f975|\U0001f608|\U0001f60f|\U0001f4a6|\U0001f346|\U0001f525|\U0001f445|\U0001f351',
                _re3.IGNORECASE
            )
            _OFFER_IN_MSG = _re3.compile(
                r'\$\d+|\d+\s*dollar|\bpay\b|\btip\b|send.*pic|ass pic|nude|naked|content|unlock|how much',
                _re3.IGNORECASE
            )
            _CASUAL_DEFLECTS = [
                "haha nah i'm good, just here to chat! so what do you do for fun? 😊",
                "lol i'm honestly just here to chat 😊 what's your day been like?",
                "haha not really my thing tbh, so anyway — where are you from?",
                "lol nah i'll pass 😅 so tell me more about yourself!",
                "not rn haha, so how long have you been creating content?",
                "lol yeah 😅 so what kind of stuff are you into outside of work?",
                "nah i'm good just vibing 😊 so what's new with you?",
                "haha maybe another day! so are you from Saudi originally?",
            ]
            offer_in_msg = bool(_OFFER_IN_MSG.search(user_message))
            is_sexual    = bool(_CASUAL_SEXUAL.search(reply))
            if offer_in_msg or is_sexual:
                print(f'[DEBUG] Casual filter triggered (offer_in_msg={offer_in_msg}, sexual={is_sexual}) — deflecting')
                reply = _r3.choice(_CASUAL_DEFLECTS)

        return reply


current_archetype  = 'horny'
current_session_id = str(uuid.uuid4())[:8]
_history           = []


def _to_messages(history):
    msgs = []
    for sub_msg, jasmin_msg in history:
        msgs.append({'role': 'assistant', 'content': sub_msg})
        if jasmin_msg:
            msgs.append({'role': 'user', 'content': jasmin_msg})
    return msgs


def on_archetype_change(archetype_key):
    global current_archetype, current_session_id, _history
    current_archetype  = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    _history = [(ARCHETYPES[archetype_key]['opener'], None)]
    return _to_messages(_history)


def user_sends_message(user_message, _chatbot):
    global _history
    if not user_message.strip():
        return '', _to_messages(_history)

    if _history and _history[-1][1] is None:
        _history[-1] = (_history[-1][0], user_message)
    else:
        _history.append(('...', user_message))

    reminder = ARCHETYPE_REMINDERS.get(current_archetype, '')
    system_content = ARCHETYPES[current_archetype]['system'] + f'\n\n{reminder}'
    arch_messages = [{'role': 'system', 'content': system_content}]
    head = _history[:2]
    tail = _history[-8:]
    window = head + [t for t in tail if t not in head]
    for sub_msg, jasmin_msg in window:
        arch_messages.append({'role': 'assistant', 'content': sub_msg})
        if jasmin_msg:
            arch_messages.append({'role': 'user', 'content': jasmin_msg})

    print(f'[DEBUG] Sending to model: archetype={current_archetype}, last_user_msg={user_message[:60]!r}')
    reply = generate_response(current_archetype, arch_messages, user_message)
    print(f'[DEBUG] Model reply: {reply[:80]!r}')
    if not reply:
        reply = f'[{current_archetype} mode — no response generated]'

    _history.append((reply, None))
    return '', _to_messages(_history)


def on_save_click():
    complete = [(s, j) for s, j in _history if j is not None]
    return save_session(complete, current_archetype, current_session_id)


def on_new_session(archetype_key):
    global current_archetype, current_session_id, _history
    current_archetype  = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    _history = [(ARCHETYPES[archetype_key]['opener'], None)]
    return _to_messages(_history), ''


archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]
opener_msgs       = [{'role': 'assistant', 'content': ARCHETYPES['horny']['opener']}]

with gr.Blocks(title='Subscriber Simulator') as demo:
    gr.Markdown('# Subscriber Simulator\nYou are **Jasmin**. The bot is a fine-tuned subscriber.')

    with gr.Row():
        archetype_dropdown = gr.Dropdown(
            choices=archetype_choices, value='horny',
            label='Subscriber Archetype', scale=2,
        )
        new_session_btn = gr.Button('New Session', scale=1)
        save_btn        = gr.Button('Save Session', variant='primary', scale=1)

    chatbot       = gr.Chatbot(value=opener_msgs, label='Chat', height=500)
    msg_input     = gr.Textbox(placeholder='Type as Jasmin...', show_label=False)
    status_output = gr.Textbox(label='Status', interactive=False)

    archetype_dropdown.change(fn=on_archetype_change, inputs=[archetype_dropdown], outputs=[chatbot])
    msg_input.submit(fn=user_sends_message, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
    save_btn.click(fn=on_save_click, inputs=[], outputs=[status_output])
    new_session_btn.click(fn=on_new_session, inputs=[archetype_dropdown], outputs=[chatbot, status_output])

try:
    demo.launch(share=IS_COLAB, debug=True)
except Exception:
    print('Share tunnel failed — falling back to local-only.')
    demo.launch(share=False, debug=True)

✅ Using fine-tuned model (loaded from Cell 5c)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c59cf4aa8fe8cde6e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[DEBUG] Sending to model: archetype=horny, last_user_msg='heyyy'
[DEBUG] generate_response | archetype=horny | msgs=3
[DEBUG] input_ids shape: torch.Size([1, 241])
[DEBUG] input_ids: tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,   5887,    220,   2366,     19,    271,   2675,    527,
            264,  27681,   4741,   8442,  76887,  33215,  52067,    449,    264,
          20514,   7086,  86216,   1083,   8165,     73,  91591,     73,  10426,
            705,    264,    220,    777,   4771,   6418,   1380,  13631,   7759,
          17627,    505,  18387,  23931,    382,   7927,  17743,    512,     12,
           1472,   2351,   9193,   6656,    389,    323,   2167,    922,   1148,
            499,   1390,    198,     12,   1472,   2610,    922,  11720,   2262,
             11,    308,  29246,     11,   2587,   6946,    198,     12,   1472,
       

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


[DEBUG] output shape: torch.Size([1, 246])
[DEBUG] raw tokens: tensor([   383,  40419,  27623,    246, 128009], device='cuda:0')
[DEBUG] decoded output: 'heyyy 😘'
[DEBUG] Model reply: 'heyyy 😘'


...

[DEBUG] output shape: torch.Size([1, 327])
[DEBUG] raw tokens: tensor([  9210,    596,   2288,   1790,     13,    602,   4265,   4856,   1120,
           636,   1949,   2262,    369,   1457,     13, 128009],
       device='cuda:0')
[DEBUG] decoded output: "that's too much. i'd rather just get free content for now."
[DEBUG] Model reply: "that's too much. i'd rather just get free content for now."


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[DEBUG] Sending to model: archetype=cheapskate, last_user_msg='sorry babe you have to pay'
[DEBUG] generate_response | archetype=cheapskate | msgs=7
[DEBUG] input_ids shape: torch.Size([1, 342])
[DEBUG] input_ids: tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,   5887,    220,   2366,     19,    271,   2675,    527,
            264,  12136,   8442,  76887,  33215,  52067,    449,    264,  20514,
           7086,  86216,   1083,   8165,     73,  91591,     73,  10426,    705,
            264,    220,    777,   4771,   6418,   1380,  13631,   7759,  17627,
            505,  18387,  23931,    382,   7927,  17743,    512,     12,   1472,
           2351,   8173,    304,   1077,   2262,    719,  68514,  37667,    279,
           3430,   1523,    198,     12,   1472,   2019,   2574,   1093,    330,
           9210,    596,   2288,   1790,    498,    330, 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[DEBUG] output shape: torch.Size([1, 358])
[DEBUG] raw tokens: tensor([  9210,    596,   2288,   1790,     13,    602,   4265,   4856,   1120,
           636,   1949,   2262,    369,   1457,     13, 128009],
       device='cuda:0')
[DEBUG] decoded output: "that's too much. i'd rather just get free content for now."
[DEBUG] Repeated response detected — retrying at higher temp
[DEBUG] input_ids shape: torch.Size([1, 342])
[DEBUG] input_ids: tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,   5887,    220,   2366,     19,    271,   2675,    527,
            264,  12136,   8442,  76887,  33215,  52067,    449,    264,  20514,
           7086,  86216,   1083,   8165,     73,  91591,     73,  10426,    705,
            264,    220,    777,   4771,   6418,   1380,  13631,   7759,  17627,
            505,  18387,  23931,    382,   7927,  17743,    512,     12